In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Search Inside Videos with Managed Video Ingestion in Agent Retrieval

This notebook introduces **managed video ingestion** in **[Agent Retrieval](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)** (formerly Vector Search 2.0): point a data store at video files in Cloud Storage, and search inside them with plain-language text queries. Each result is a moment in a video, with its start and end time. It is written for developers who are familiar with vector search and embeddings concepts, but new to this Google Cloud service.

**Searching documents instead of videos?** See [Search Your Documents with SearchDocuments in Agent Retrieval](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb). To learn the underlying Collections, Data Objects, and search APIs, see [Introduction to Agent Platform Vector Search 2.0](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/vector-search-2-intro.ipynb).

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-video-search.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fembeddings%2Fagent-retrieval-video-search.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/embeddings/agent-retrieval-video-search.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-video-search.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-video-search.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-video-search.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-video-search.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-video-search.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-video-search.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author |
| --- |
| [Kaz Sato](https://github.com/kazunori279) |

## What is Managed Video Ingestion?

Making video searchable usually means building a pipeline: transcode the files, cut them into clips, embed each clip with a multimodal model, write the embeddings to a vector database, and keep track of which clip came from which second of which video. Managed video ingestion does all of that for you:

- **Point a data store at your videos**: Create a `VIDEO` data store and import video files from Cloud Storage (MP4, MOV, WebM, MKV, AVI, WMV, MPEG-2 TS, MXF).
- **The service does the rest**: It transcodes each video, splits it into fixed-length **segments** (30 seconds by default), embeds each segment's frames (and optionally its audio) with `gemini-embedding-2`, and writes the segments into an Agent Retrieval Collection it creates for you.
- **Search with text**: Send a text query to the data store with **SearchDocuments** and get back the matching segments, each with its source video and its start and end time. You don't need a transcript, captions, or tags.

### What changes compared to building it yourself

| Without managed ingestion | With managed ingestion |
|---|---|
| Transcode videos and cut them into clips | Import the files; segmentation is one setting (`segment_duration`) |
| Call a multimodal embedding model for every clip | Embeddings are generated with `gemini-embedding-2` |
| Design a Collection schema for clips, offsets, and source URIs | The data store creates the Collection and its schema |
| Track which second of which video each clip came from | Every result carries its source URI and `start_offset` / `end_offset` |
| Write a separate pipeline for speech (transcription) | Set `audio_mode: ENABLED` to embed the audio track with the frames |

### Core Concepts

1. **Data store**: The ingestion resource. It holds your videos and the configuration that says how to segment and embed them. A data store handles one modality, fixed at creation: this notebook uses `VIDEO`.
2. **Document**: One video file from Cloud Storage, with optional metadata you attach to it (for example, `category`).
3. **Segment**: A fixed-length window of a video with its own embedding. Searches return segments, so each result points to a moment, not a whole video.
4. **Backing Collection**: The Agent Retrieval Collection the data store creates and writes segments into. SearchDocuments queries it for you, and you can also query it directly.

> **Note:** The data store and SearchDocuments APIs are served at `v1alpha` and are not yet in the `google-cloud-vectorsearch` Python SDK. This notebook calls them over REST with an authorized session.

Let's explore each step with hands-on examples!

## Example Scenario: Searching a Video Library

To demonstrate managed video ingestion, we'll build **moment search over a small library of Google videos**: a user describes a scene or something that was said, and gets back the video and the exact time range where it happens, ready to play.

**For this demo**: We'll ingest **5 videos (about 19 minutes in total)** from the public `cloud-samples-data` bucket. They mix product ads, interviews, a keynote, and nature footage, and all of them have narration or dialogue.

| Document ID | Video | Length | category |
|---|---|---|---|
| `pixel-guided-frame` | Behind the scenes of a Pixel ad directed by a blind filmmaker | 5:23 | `behind_the_scenes` |
| `ml-solving-problems` | Machine Learning: Solving Problems Big, Small, and Prickly | 5:19 | `technology` |
| `google-sustainability` | Sundar Pichai on Google's sustainability work | 3:58 | `keynote` |
| `jane-goodall-gombe` | Gombe National Park, narrated by Jane Goodall | 2:43 | `nature` |
| `google-trips` | Introducing Google Trips (ad) | 1:49 | `product_ad` |

### Business Use Cases:

1. **Moment search**: "a mother chimpanzee grooming her baby" returns the video and the 20 seconds where it happens
2. **Filtered search**: "Only search the product ads" or "only search the first 20 seconds of each video"
3. **Search within one video**: Jump to a scene inside a video the user is already watching
4. **Search what was said**: Find the moment a speaker mentions a topic, even when the picture shows something else

Throughout this notebook, we'll walk through each step: staging the videos, creating data stores, importing the videos, and then searching them. Let's get started!

-----

## Prerequisites

This tutorial requires a Google Cloud project that is linked with a billing account. To create a new project, take a look at [this document](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/start/cloud-environment) to create a project and setup a billing account for it.
To get the permissions that you need to give a service account access to enable APIs and interact with Agent Platform resources, ask your administrator to grant you the [Security Admin](https://cloud.google.com/iam/docs/roles-permissions/iam#iam.securityAdmin) (`roles/iam.securityAdmin`) IAM role on your project. For more information about granting roles, see [Manage access to projects, folders, and organizations](https://cloud.google.com/iam/docs/granting-changing-revoking-access).

### Important: Resource Cleanup

Agent Retrieval resources incur costs when active. Make sure to run the cleanup section at the end of this tutorial to delete the data stores, their backing Collections, and the Cloud Storage bucket.

## Install the Libraries

We'll call the data store and SearchDocuments APIs over REST with `google-auth` and `requests`. We'll also install the `google-cloud-vectorsearch` SDK to look at the backing Collection later.

In [ ]:
%pip install google-auth requests google-cloud-vectorsearch

Then, set `PROJECT_ID` for your Google Cloud project. Managed ingestion runs in **`us-east4`** and **`europe-west4`** only, so your data stores must be in one of them. You can leave `LOCATION` as the default value.

In [ ]:
# Set PROJECT_ID and LOCATION
PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "us-east4"  # @param ["us-east4", "europe-west4"]

# Validate PROJECT_ID is set
if PROJECT_ID == "your-project-id" or not PROJECT_ID or PROJECT_ID == "":
    raise ValueError(
        "⚠️ Please set PROJECT_ID to your actual Google Cloud project ID in the cell above"
    )

print(f"✅ Using project: {PROJECT_ID}")
print(f"✅ Using location: {LOCATION}")

## Authentication

On Colab, run the following to authenticate calls to the Agent Retrieval APIs. For Colab Enterprise and Cloud Workbench, you can skip this part.

In [ ]:
import sys

# Additional authentication is required for Google Colab
if "google.colab" in sys.modules:
    # Authenticate user to Google Cloud
    from google.colab import auth

    auth.authenticate_user()

## Enable APIs

Run the following command to enable the APIs used in this notebook:

- `vectorsearch.googleapis.com`: Agent Retrieval (data stores, Collections, and search)
- `aiplatform.googleapis.com`: The embedding model that embeds your videos and queries

In [ ]:
! gcloud services enable vectorsearch.googleapis.com aiplatform.googleapis.com --project "{PROJECT_ID}"

## Grant the Agent Retrieval Service Agent Access

Managed ingestion reads your videos and writes reports with the Agent Retrieval **service agent**, `service-{PROJECT_NUMBER}@gcp-sa-vectorsearch.iam.gserviceaccount.com`. The following cell makes sure the service agent exists and grants it the `roles/vectorsearch.serviceAgent` role on your project.

In [ ]:
PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]
SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-vectorsearch.iam.gserviceaccount.com"

! gcloud beta services identity create --service=vectorsearch.googleapis.com --project={PROJECT_ID}
! gcloud projects add-iam-policy-binding {PROJECT_ID} --member="serviceAccount:{SERVICE_AGENT}" --role="roles/vectorsearch.serviceAgent" --condition=None --quiet > /dev/null

print(f"✅ Granted roles/vectorsearch.serviceAgent to {SERVICE_AGENT}")

-----

# Part 1: Stage Your Videos in Cloud Storage

Managed ingestion reads videos from Cloud Storage. **One Cloud Storage object becomes one document**, and each document becomes many searchable segments.

**What you'll accomplish:**
- Create a Cloud Storage bucket for the import manifests and reports
- Write a JSONL manifest that lists each video with its ID and metadata

## Supported Videos

- **Formats**: MP4, MOV, WebM, MKV, AVI, WMV, MPEG-2 TS, and MXF (OP1A). The format is detected from the file's content, not its extension; both the container and the video codec must be readable by the [Transcoder API](https://docs.cloud.google.com/transcoder/docs/concepts/supported-input-and-output-formats).
- **Limits**: Up to 100 videos per import, 15 hours per video, and 600 GB per file. Files with no video stream (for example, audio-only files) are rejected.

## Two Ways to Point an Import at Your Videos

1. **URI patterns**: Pass patterns such as `gs://my-bucket/videos/*`. Each matched file becomes a document, and its ID is derived from its path. Wildcards don't recurse into subfolders.
2. **JSONL manifest**: Pass one JSONL file that lists each video on its own line, with the document ID you choose, the file's `gcsUri`, and optional metadata. **Use this when you want metadata to filter on or return in search.**

We'll use a JSONL manifest so that every video carries a `category` and a human-readable `title`. The videos themselves stay in the public `cloud-samples-data` bucket; only the manifests and the import reports go into your own bucket.

## Create a Cloud Storage Bucket

First, let's pick IDs for the data stores and the bucket. We'll create **two data stores** from the same videos, one that embeds the audio track and one that embeds frames only, so we can compare them in Part 5.

In [ ]:
from datetime import datetime

suffix = datetime.now().strftime("%m%d%y-%H%M%S")
video_store_id = f"sample-videos-{suffix}"  # Frames + audio
frames_store_id = f"sample-videos-frames-{suffix}"  # Frames only
BUCKET_NAME = f"{PROJECT_ID}-videos-{suffix}"

print(f"Data stores: {video_store_id}, {frames_store_id}")
print(f"Bucket: gs://{BUCKET_NAME}")

In [ ]:
! gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --project={PROJECT_ID}

## Write the JSONL Manifests

Each line of a manifest is a video:

- `name`: The full resource name of the document. **The last part becomes the document ID**, so you choose the IDs. The name includes the data store ID, so we write one manifest per data store.
- `content.gcsUri`: The Cloud Storage URI of the video.
- `metadata.value`: The video's metadata. It must match the metadata schema we declare when we create the data stores.

In [ ]:
import json
import subprocess

SOURCE_PREFIX = "gs://cloud-samples-data"

# Document ID -> (file path, metadata)
videos = {
    "pixel-guided-frame": (
        "generative-ai/video/behind_the_scenes_pixel.mp4",
        {"category": "behind_the_scenes", "title": "Behind the scenes of a Pixel ad"},
    ),
    "ml-solving-problems": (
        "video/Machine Learning Solving Problems Big, Small, and Prickly.mp4",
        {"category": "technology", "title": "Machine Learning: Solving Problems"},
    ),
    "google-sustainability": (
        "generative-ai/video/google_sustainability.mp4",
        {"category": "keynote", "title": "Sundar Pichai on sustainability"},
    ),
    "jane-goodall-gombe": (
        "video/JaneGoodall.mp4",
        {"category": "nature", "title": "Gombe National Park with Jane Goodall"},
    ),
    "google-trips": (
        "generative-ai/video/introducing_google_trips.webm",
        {"category": "product_ad", "title": "Introducing Google Trips"},
    ),
}


def write_manifest(data_store_id: str) -> str:
    """Write a JSONL manifest for one data store and upload it to the bucket."""
    file_name = f"{data_store_id}.jsonl"
    with open(file_name, "w") as f:
        for doc_id, (path, metadata) in videos.items():
            line = {
                "name": f"projects/{PROJECT_ID}/locations/{LOCATION}/dataStores/{data_store_id}/documents/{doc_id}",
                "content": {"gcsUri": f"{SOURCE_PREFIX}/{path}"},
                "metadata": {"value": metadata},
            }
            f.write(json.dumps(line) + "\n")
    manifest_uri = f"gs://{BUCKET_NAME}/manifests/{file_name}"
    subprocess.run(
        ["gcloud", "storage", "cp", file_name, manifest_uri, "--quiet"], check=True
    )
    return manifest_uri


manifest_uris = {
    video_store_id: write_manifest(video_store_id),
    frames_store_id: write_manifest(frames_store_id),
}

print(f"✅ Wrote manifests for {len(videos)} videos")
! head -n 1 {video_store_id}.jsonl

-----

# Part 2: Create the Data Stores and Import the Videos

Now that the manifests are in place, let's create the **data stores** and import the videos into them.

**What you'll accomplish:**
- Set up an authorized REST session for the `v1alpha` API
- Choose the segment duration and audio mode
- Declare a metadata schema
- Create two data stores and import the videos into both

## Set Up the REST Session

The data store and SearchDocuments APIs are not yet in the Python SDK, so we call them with an `AuthorizedSession` from `google-auth`, which attaches your credentials to every request. We also define two small helpers:

- `check()`: Raises an error with the API's message when a call fails
- `wait()`: Polls a long-running operation (LRO) until it finishes

In [ ]:
import time

import google.auth
import requests
from google.auth.transport.requests import AuthorizedSession

credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
session = AuthorizedSession(credentials)

API_BASE = "https://vectorsearch.googleapis.com/v1alpha"
PARENT_URL = f"{API_BASE}/projects/{PROJECT_ID}/locations/{LOCATION}"


def data_store_url(data_store_id: str) -> str:
    return f"{PARENT_URL}/dataStores/{data_store_id}"


def check(response: requests.Response) -> dict:
    """Return the JSON body, or raise an error with the API's message."""
    if not response.ok:
        raise RuntimeError(f"{response.status_code}: {response.text}")
    return response.json()


def wait(operation: dict, poll_seconds: int = 10) -> dict:
    """Poll a long-running operation until it finishes, then return it."""
    start = time.time()
    while True:
        op = check(session.get(f"{API_BASE}/{operation['name']}"))
        if op.get("done"):
            if "error" in op:
                raise RuntimeError(op["error"])
            return op
        elapsed = int(time.time() - start)
        print(f"   Still running... ({elapsed // 60} min {elapsed % 60} sec elapsed)")
        time.sleep(poll_seconds)

## Processing Options for Videos

A **data store** holds your videos and the **processing configuration** that controls how they are segmented and embedded. Only `document_type` is required; everything else has a default.

| Setting | Options | Default |
|---|---|---|
| **Segment duration** (`segmentation_config.fixed_window_config.segment_duration`) | More than 5 seconds, up to 80 seconds, e.g. `"20s"` | `30s` |
| **Audio** (`embedding_config.audio_mode`) | `ENABLED`: embed the first audio track together with the frames<br>`DISABLED`: embed frames only | `DISABLED` |
| **Embedding model** | Not settable | `gemini-embedding-2`, 768 dimensions |

### Choosing a Segment Duration

The segment duration decides how precise the timestamps are. **Short windows (10-20 seconds)** give tight answers and suit queries about a specific moment, like a product shot or a single action. **Long windows (60-80 seconds)** give each embedding more context, which helps with thematic queries, and produce fewer segments to store and search. We'll use **20 seconds**, since our videos are short and our queries target specific scenes. To change it later, update the data store and re-import the videos.

### Turning Audio On

With `audio_mode: DISABLED`, only the frames are embedded, so a query matches what is **shown**. Set it to `ENABLED` when people will search for what is **said**: talks, interviews, narration, support calls. The audio is embedded into the same vector as the frames; there is no separate transcript field and you query it the same way. Only the first audio track is used.

### Metadata Schema

To filter on or return per-video metadata, declare it in `document_metadata_schema` as a [JSON Schema](https://json-schema.org/), the same way as for document data stores.

## Create the Data Stores

Let's create both data stores with the same settings, except for `audio_mode`.

In [ ]:
def create_video_data_store(data_store_id: str, audio_mode: str) -> dict:
    """Create a VIDEO data store and return the create operation."""
    return check(
        session.post(
            f"{PARENT_URL}/dataStores",
            params={"data_store_id": data_store_id},
            json={
                "display_name": f"Sample videos (audio {audio_mode.lower()})",
                "document_type": "VIDEO",  # UNSTRUCTURED or VIDEO. Immutable.
                "processing_config": {
                    "video_processing_config": {
                        # One segment every 20 seconds
                        "segmentation_config": {
                            "fixed_window_config": {"segment_duration": "20s"}
                        },
                        # ENABLED: frames + first audio track. DISABLED: frames only.
                        "embedding_config": {"audio_mode": audio_mode},
                    }
                },
                # Metadata properties we can filter on and return in search
                "document_metadata_schema": {
                    "schema": {
                        "type": "object",
                        "properties": {
                            "category": {"type": "string"},
                            "title": {"type": "string"},
                        },
                    }
                },
            },
        )
    )


print("⏳ Creating data stores...")
wait(create_video_data_store(video_store_id, "ENABLED"))
wait(create_video_data_store(frames_store_id, "DISABLED"))
print("✅ Data stores created!")

video_store = check(session.get(data_store_url(video_store_id)))
print(f"\nState:           {video_store['state']}")
print(f"Collection name: {video_store['collectionName']}")
print("Processing config:")
print(json.dumps(video_store["processingConfig"], indent=2))

### Key Points:

What we just accomplished:

✅ **Created two `VIDEO` data stores** with 20-second segments  
✅ **Turned audio on** for `video_store_id` and left it off for `frames_store_id`  
✅ **Declared a metadata schema** with two properties: `category` and `title`  
✅ **Got a backing Collection for each data store**, with no schema to design

## Import the Videos

Now let's import the videos listed in the manifests into both data stores. The import options are the same as for documents:

- `source_format: SOURCE_FORMAT_DOCUMENT_JSON` tells the import that the URI is a JSONL manifest, not a video file.
- `import_output_config.error_gcs_prefix` is where per-file failures are written. **It must point to an empty Cloud Storage folder**, so use a new prefix for each import.

Only one import can run at a time per data store, but different data stores import in parallel, so we start both imports before waiting.

In [ ]:
import_ops = {}
for data_store_id, manifest_uri in manifest_uris.items():
    error_prefix = f"gs://{BUCKET_NAME}/import-errors/{data_store_id}/{datetime.now().strftime('%Y%m%d-%H%M%S')}/"
    import_ops[data_store_id] = check(
        session.post(
            f"{data_store_url(data_store_id)}:importDocuments",
            json={
                "gcs_source": {
                    "input_uris": [manifest_uri],
                    "source_format": "SOURCE_FORMAT_DOCUMENT_JSON",
                },
                "import_output_config": {"error_gcs_prefix": error_prefix},
            },
        )
    )
    print(f"✅ Import started for {data_store_id}")

### Waiting for the Imports

Video imports take much longer than document imports: every video is transcoded, then embedded segment by segment. For these 5 videos, the two imports together typically take **around 20-25 minutes**. When two video imports run at the same time in a project, one of them can finish well before the other.

If you hit Colab's cell execution timeout, simply re-run the cell to continue waiting.

In [ ]:
print("⏳ Waiting for both imports to finish...")
start = time.time()
pending = dict(import_ops)
while pending:
    for data_store_id, op in list(pending.items()):
        op = check(session.get(f"{API_BASE}/{op['name']}"))
        if op.get("done"):
            if "error" in op:
                raise RuntimeError(op["error"])
            print(
                f"✅ {op['metadata'].get('successCount', 0)} videos imported into {data_store_id}"
            )
            del pending[data_store_id]
    if pending:
        elapsed = int(time.time() - start)
        print(f"   Still running... ({elapsed // 60} min {elapsed % 60} sec elapsed)")
        time.sleep(60)

### Listing the Documents

Let's list the documents in the data store with audio, along with their metadata. Each document has a `state` of `SUCCEEDED` once it has been processed.

In [ ]:
docs = check(
    session.get(f"{data_store_url(video_store_id)}/documents", params={"pageSize": 100})
)

for doc in sorted(docs["documents"], key=lambda d: d["name"]):
    doc_id = doc["name"].split("/")[-1]
    print(f"{doc_id:24} {doc['state']:10} {doc.get('metadata', {}).get('value')}")

-----

# Part 3: Search Inside the Videos

With the videos ingested, let's search them! We use the same **SearchDocuments** API as for documents.

**What you'll accomplish:**
- Run a text search against the video data store
- Understand the result structure: segments and their time offsets
- Turn a result into a link and a player that starts at the right second

## Your First Search

A SearchDocuments request needs only the data store and a text query. For a `VIDEO` data store, SearchDocuments runs a **semantic search**: it embeds your query text with the same model that embedded the video segments, and returns the closest segments.

**Scenario**: A user looks for "a mother chimpanzee grooming her baby".

In [ ]:
def search_url(data_store_id: str) -> str:
    return f"{data_store_url(data_store_id)}/documents:search"


response = check(
    session.post(
        search_url(video_store_id),
        json={
            "text_query": {"text": "a mother chimpanzee grooming her baby"},
            "top_k": 3,
        },
    )
)

print(json.dumps(response["results"][0], indent=2))

### Understanding the Results

Each result is a **segment** of a video:

- `document.name`: The video the segment came from. Several results can point to the same video, often consecutive segments of one scene.
- `document.content.gcsUri`: The original video file in Cloud Storage.
- `videoDetails.startOffset` / `endOffset`: Where the segment starts and ends in the source video, as a duration string in seconds (for example `"60s"`). The last segment of a video ends at the video's length, so its end offset can have a fraction.

Let's write two helpers: one that prints results compactly with `mm:ss` times and a link that starts playback at the segment, and one that shows a video player for a single segment. The links and players work here because `cloud-samples-data` is a public bucket; for your own private videos, generate [signed URLs](https://docs.cloud.google.com/storage/docs/access-control/signed-urls) instead.

In [ ]:
import urllib.parse

from IPython.display import HTML, display

def seconds(offset: str) -> float:
    """Convert a duration string such as "60s" to seconds."""
    return float(offset.rstrip("s"))


def mmss(value: float) -> str:
    return f"{int(value) // 60:02d}:{int(value) % 60:02d}"


def public_url(gcs_uri: str) -> str:
    """HTTPS URL of an object in a public Cloud Storage bucket."""
    return "https://storage.googleapis.com/" + urllib.parse.quote(
        gcs_uri[len("gs://") :]
    )


def print_results(response: dict) -> None:
    """Print SearchDocuments results, one segment per entry."""
    for i, result in enumerate(response.get("results", []), 1):
        document = result["document"]
        start = seconds(result["videoDetails"]["startOffset"])
        end = seconds(result["videoDetails"]["endOffset"])
        doc_id = document["name"].split("/")[-1]
        metadata = document.get("metadata", {}).get("value")
        link = f"{public_url(document['content']['gcsUri'])}#t={int(start)}"

        print(
            f"{i:2}. {doc_id:22} {mmss(start)}-{mmss(end)}"
            + (f"  {metadata}" if metadata else "")
        )
        print(f"    {link}")


def play(result: dict) -> None:
    """Show a video player that plays only the segment of a search result."""
    start = seconds(result["videoDetails"]["startOffset"])
    end = seconds(result["videoDetails"]["endOffset"])
    url = public_url(result["document"]["content"]["gcsUri"])
    display(
        HTML(
            f'<video src="{url}#t={start:.0f},{end:.0f}" controls width="480"></video>'
        )
    )


print_results(response)
play(response["results"][0])

The top result is the segment of the Jane Goodall video where the mother chimpanzee grooms her infant (around 01:19). Press play: the player starts at the segment and stops at its end, using a [media fragment](https://www.w3.org/TR/media-frags/) (`#t=start,end`) on the video URL.

## Searching for Scenes

Let's try a few more scene descriptions. Each query names something visual, and we print the top result for each.

In [ ]:
queries = [
    "taking a selfie in front of the Sagrada Familia",
    "cucumbers being sorted by a machine",
    "a phone camera framing a dog outdoors",
    "an aerial view of a large lake and mountains",
]

for query in queries:
    response = check(
        session.post(
            search_url(video_store_id), json={"text_query": {"text": query}, "top_k": 1}
        )
    )
    print(f"🔎 {query}")
    print_results(response)
    print()

Each query lands on a different video and on the 20 seconds that show the scene: the selfie in the Google Trips ad, the cucumber sorter in the machine learning video, the Pixel Guided Frame demo with a dog, and the aerial view of Lake Tanganyika in the Gombe video. None of these videos has tags or captions; the matches come from the embeddings of the frames.

Several consecutive segments of the same video often match one query. If you want one result per scene rather than one per segment, group the results by `document.name` and keep the earliest `startOffset` of each run.

-----

# Part 4: Filter by Metadata and Time Range

As with documents, pass `filter` to search only the segments that match. For a `VIDEO` data store, you can filter on:

| Path | Notes |
|---|---|
| `metadata.<property>` | Must be declared in the data store's `document_metadata_schema` |
| `video_details.start_offset` / `end_offset` | In seconds. Fractions are allowed. |

The operators are the same: `$eq`, `$ne`, `$lt`, `$lte`, `$gt`, `$gte`, `$in` (strings), `$nin`, `$exists`, combined with `$and`, `$or`, and `$not`.

### Example 1: Filter by Metadata

**Scenario**: "someone taking a photo with a smartphone" matches the Pixel video best, since it is all about the Pixel camera. Let's search **only the product ads**.

In [ ]:
query = "someone taking a photo with a smartphone"

for name, search_filter in [
    ("All videos", None),
    ("Product ads only", {"metadata.category": {"$eq": "product_ad"}}),
]:
    request = {"text_query": {"text": query}, "top_k": 3}
    if search_filter:
        request["filter"] = search_filter
    response = check(session.post(search_url(video_store_id), json=request))
    print(f"{name}:")
    print_results(response)
    print()

Without the filter, the Pixel video takes the top results. With the filter, only the Google Trips ad is searched, and its selfie scene comes first.

### Example 2: Filter by Time Range and Return Metadata

**Scenario**: Show the **opening** of each video. We search only segments that start in the first 20 seconds, and return each video's metadata with `output_fields.metadata` so we can show its title.

In [ ]:
response = check(
    session.post(
        search_url(video_store_id),
        json={
            "text_query": {"text": "the opening scene of a video"},
            "top_k": 5,
            "filter": {"video_details.start_offset": {"$lt": 20}},
            "output_fields": {"metadata": ["title", "category"]},
        },
    )
)

print_results(response)

Every result is the first segment of a video, and each one now carries `document.metadata.value` with the requested properties.

### Example 3: Search Within One Video

When the user is already watching a video, pass `document_ids` to search only inside it, like an in-video search box.

**Scenario**: While watching the Google Trips ad, jump to "live flamenco music at a restaurant".

In [ ]:
response = check(
    session.post(
        search_url(video_store_id),
        json={
            "text_query": {"text": "live flamenco music at a restaurant"},
            "top_k": 3,
            "document_ids": ["google-trips"],
        },
    )
)

print_results(response)
play(response["results"][0])

### Things to Know

- **Offsets are in seconds** in filters and results. Write `{"video_details.start_offset": {"$gte": 60}}` for "after the first minute".
- **Page-range filters don't apply.** `unstructured_details.page_range` is for document data stores; using it on a video data store returns `INVALID_ARGUMENT`.
- **Reranking isn't available for video.** `ranking_spec.reranking_spec.enabled` returns `INVALID_ARGUMENT` on a `VIDEO` data store.

-----

# Part 5: Search What Was Said with Audio Mode

So far we searched for things that are **shown**. Many videos also carry information that is only **said**: in a keynote or an interview, the picture often shows the speaker while they talk about something else.

Our two data stores hold the same videos with the same segments. The only difference is that `video_store_id` embedded the audio track and `frames_store_id` did not. Let's ask both about things Sundar Pichai **says** in the sustainability keynote while the picture shows him talking:

- At 01:41: "our next-generation geothermal project in Nevada"
- At 03:20: reducing emissions by an amount "equal to the emissions of a major country like Japan"

In [ ]:
queries = [
    "geothermal project in Nevada",
    "emissions equal to a major country like Japan",
]

for query in queries:
    print(f"🔎 {query}")
    for name, data_store_id in [
        ("Frames only  ", frames_store_id),
        ("Frames + audio", video_store_id),
    ]:
        response = check(
            session.post(
                search_url(data_store_id),
                json={"text_query": {"text": query}, "top_k": 1},
            )
        )
        result = response["results"][0]
        start = seconds(result["videoDetails"]["startOffset"])
        end = seconds(result["videoDetails"]["endOffset"])
        doc_id = result["document"]["name"].split("/")[-1]
        print(f"   {name}: {doc_id} {mmss(start)}-{mmss(end)}")
    print()

With frames only, the top results point to the wrong part of the keynote: the frames of a speaker talking look much the same from one segment to the next. With audio, each query lands on the segment where the words are spoken (01:40-02:00 and 03:20-03:40).

Let's play the segment the audio-enabled data store found for the second query:

In [ ]:
response = check(
    session.post(
        search_url(video_store_id),
        json={
            "text_query": {"text": "emissions equal to a major country like Japan"},
            "top_k": 1,
        },
    )
)
play(response["results"][0])

### When to Turn Audio On

- **Turn it on** when the spoken content matters: talks, interviews, tutorials, meetings, narrated content.
- **Leave it off** when the audio carries nothing users would search for: silent screen recordings, security footage, b-roll over music.

To change the setting for existing videos, update the data store's `processing_config` and re-import them.

-----

# Part 6: Under the Hood: The Backing Collection

Like for documents, each video data store writes its segments into a regular Agent Retrieval Collection. You can query it directly with the `google-cloud-vectorsearch` SDK, as in the [Introduction to Vector Search 2.0](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/vector-search-2-intro.ipynb) notebook. To do so, you need to know:

- **The Collection name**, from the data store's `collectionName`
- **The vector field**, `multimodal_embedding`, and the task type `RETRIEVAL_QUERY`
- **The stored field names**: fields the service writes are prefixed with `google_`, and the offsets are stored in **nanoseconds** (`google_video_details.start_offset_nanos` / `end_offset_nanos`); your metadata properties keep their declared names (`category`, `title`)
- **`output_fields`**: without it, you get back only the object name and distance

The server embeds your query text with your credentials, so this call also needs the `roles/aiplatform.user` role on the project.

In [ ]:
from google.cloud import vectorsearch_v1

data_object_search_service_client = vectorsearch_v1.DataObjectSearchServiceClient()

request = vectorsearch_v1.SearchDataObjectsRequest(
    parent=video_store["collectionName"],  # The backing Collection, not the data store
    semantic_search=vectorsearch_v1.SemanticSearch(
        search_text="a mother chimpanzee grooming her baby",
        search_field="multimodal_embedding",
        task_type=vectorsearch_v1.EmbeddingTaskType.RETRIEVAL_QUERY,
        top_k=3,
        filter={"category": {"$eq": "nature"}},  # Stored name, not metadata.category
        output_fields=vectorsearch_v1.OutputFields(
            data_fields=[
                "google_document_id",
                "google_customer_uri",
                "google_video_details.start_offset_nanos",
                "google_video_details.end_offset_nanos",
                "title",
            ]
        ),
    ),
)

for result in data_object_search_service_client.search_data_objects(request=request):
    data = result.data_object.data
    details = data["google_video_details"]
    start = details["start_offset_nanos"] / 1e9  # Nanoseconds to seconds
    end = details["end_offset_nanos"] / 1e9
    print(
        f"{data['google_document_id']} {mmss(start)}-{mmss(end)} "
        f"({data['title']}, distance={result.distance:.3f})"
    )

Querying the Collection directly gives you full control, for example to create an ANN index for a large video library or to combine several searches in one batch. For everyday moment search, SearchDocuments returns the same segments with offsets already in seconds and the source URI attached.

-----

# Part 7: Clean Up

## Important: Delete Resources to Avoid Costs

Agent Retrieval resources incur costs when active. To avoid unexpected charges, delete:

1. **Both data stores**, their documents, and their backing Collections
2. **The Cloud Storage bucket** with the manifests and import reports

## Step 1: Delete the Data Stores

A data store that still holds documents can be deleted only with `force=true`. Add `backing_collection_deleted=true` to also delete the backing Collection and its Data Objects. Without it, the `google-{data_store_id}` Collection is left behind and keeps incurring costs.

Deletion runs in the background and can take around 20 minutes. You don't need to wait for it: a data store shows the `DELETING` state while it runs, and returns `404 NOT_FOUND` once it's gone.

In [ ]:
for data_store_id in [video_store_id, frames_store_id]:
    delete_op = check(
        session.delete(
            data_store_url(data_store_id),
            params={"force": "true", "backing_collection_deleted": "true"},
        )
    )
    state = check(session.get(data_store_url(data_store_id)))["state"]
    print(f"🗑️ Deleting {data_store_id} and its backing Collection ({state})")

## Step 2: Delete the Cloud Storage Bucket

The videos stay in the public bucket; this deletes only your bucket with the manifests and import reports.

In [ ]:
! gcloud storage rm --recursive gs://{BUCKET_NAME} --quiet
print(f"🗑️ Deleted bucket: gs://{BUCKET_NAME}")
print(
    "\n✅ Cleanup complete! The data stores and their backing Collections finish deleting in the background."
)

-----

# Summary

Congratulations! You've completed an introduction to **managed video ingestion in Agent Retrieval**. Let's recap what you've learned.

## What You've Learned

### 1. **Managed Video Ingestion** - From Video Files to Searchable Moments
- Staged videos with a JSONL manifest that sets document IDs and metadata
- Created `VIDEO` data stores with 20-second segments and a metadata schema
- Imported videos with a single long-running operation per data store; the service transcoded, segmented, and embedded them

### 2. **Moment Search with SearchDocuments**
- Searched videos with plain-language scene descriptions, without transcripts, captions, or tags
- Read results as segments with start and end offsets, and turned them into links and players that start at the right second

### 3. **Narrowing the Search**
- Filtered on metadata (`metadata.category`) and time ranges (`video_details.start_offset`)
- Returned video metadata with `output_fields.metadata`
- Searched inside a single video with `document_ids`

### 4. **Audio Mode**
- Compared frames-only and frames + audio embeddings, and found spoken statements only with audio enabled

### 5. **Under the Hood**
- Queried the backing Collection directly with the SDK, using the stored `google_` field names and nanosecond offsets

## Next Steps

- [Search Your Documents with SearchDocuments in Agent Retrieval](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb): The same workflow for PDFs, Word, PowerPoint, Excel, and text files, with page-range filters and reranking
- [Introduction to Agent Platform Vector Search 2.0](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/vector-search-2-intro.ipynb): Learn the Collection, Data Object, search, and ANN index APIs under managed ingestion
- [Search documents in a data store](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search-data-stores): Data store and managed ingestion reference
- Try your own videos: put them in a bucket and point an import at `gs://your-bucket/your-prefix/*`

-----

# Appendix: Video Ingestion Reference

## Video Processing Settings

| Field | Default | Notes |
|---|---|---|
| `document_type` | | `VIDEO`. Required and immutable. |
| `video_processing_config.segmentation_config.fixed_window_config.segment_duration` | `30s` | More than 5 seconds, up to 80 seconds. Use whole seconds. |
| `video_processing_config.embedding_config.audio_mode` | `DISABLED` | `ENABLED` embeds the first audio track with the frames |
| `document_metadata_schema` | None | JSON Schema for per-video metadata |

## Limits

| Limit | Value |
|---|---|
| Regions | `us-east4`, `europe-west4` |
| Formats | MP4, MOV, WebM, MKV, AVI, WMV, MPEG-2 TS, MXF (OP1A) |
| Videos per import | 100 |
| Video duration | 15 hours |
| Video file size | 600 GB |
| Embedding model | `gemini-embedding-2`, 768 dimensions |

## SearchDocuments on a Video Data Store

| Field | Notes |
|---|---|
| `text_query.text` | Required. Matched against the segment embeddings (semantic search). |
| `top_k` | Default 10, max 100 |
| `filter` | `metadata.<property>` and `video_details.start_offset` / `end_offset` (seconds) |
| `document_ids` | Search only these videos. Max 100. |
| `output_fields.metadata` | Metadata properties to return, or `"*"` for all |
| `ranking_spec.reranking_spec.enabled` | Not supported for `VIDEO` data stores |

Each result has `document.name`, `document.content.gcsUri`, requested `document.metadata.value`, and `videoDetails.startOffset` / `endOffset` (duration strings in seconds).

## Common Errors

| Condition | Code | What to do |
|---|---|---|
| `segment_duration` of 5 seconds or less, or over 80 seconds | `INVALID_ARGUMENT` | Use a value from `6s` to `80s` |
| Empty `segmentation_config` | `INVALID_ARGUMENT` | Set `fixed_window_config` inside it, or omit the block for the default |
| Page-range filter on a video data store | `INVALID_ARGUMENT` | Use `video_details.start_offset` / `end_offset` |
| Reranking requested on a video data store | `INVALID_ARGUMENT` | Remove `ranking_spec` |
| Spoken content doesn't match | `OK` | `audio_mode` is `DISABLED`; enable it and re-import |
| A file was skipped | `OK` (reported in error files) | Check that it has a video stream and a codec the Transcoder API reads |